In [ ]:

# ER Band: Healthcare AI Prototype
# Initial setup and environment checks

import sys
import pandas as pd
import numpy as np
from pathlib import Path

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

print("\nNotebook environment is ready.")


In [ ]:

# Upload and inspect the official demo dataset

from google.colab import files
import zipfile

uploaded = files.upload()

zip_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_names:
    raise ValueError(
        "Please upload the MIMIC-IV-ED Demo ZIP file."
    )

zip_path = Path(zip_names[0])
extract_dir = Path("/content/mimic_ed_demo")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_dir)

print("Dataset extracted successfully.")
print("\nFiles found:")

for path in sorted(extract_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(extract_dir))


In [ ]:

# ER Band — Step 4: Inspect the emergency department data

import pandas as pd
from pathlib import Path

# Folder created in the previous step
data_dir = Path(
    "/content/mimic_ed_demo/mimic-iv-ed-demo-2.2/ed"
)

# Load the three tables we want to examine first
edstays = pd.read_csv(data_dir / "edstays.csv.gz")
triage = pd.read_csv(data_dir / "triage.csv.gz")
vitalsign = pd.read_csv(data_dir / "vitalsign.csv.gz")

# 1. Check the size and columns of each table
for name, table in [
    ("ED stays", edstays),
    ("Triage", triage),
    ("Vital signs", vitalsign)
]:
    print(f"\n{'=' * 50}")
    print(f"{name}: {table.shape[0]} rows, {table.shape[1]} columns")
    print("Columns:", table.columns.tolist())

# 2. Check missing data
print("\nMISSING DATA (%)")
for name, table in [
    ("ED stays", edstays),
    ("Triage", triage),
    ("Vital signs", vitalsign)
]:
    print(f"\n{name}")
    print((table.isna().mean() * 100).round(1).to_string())

# 3. Check whether each table has the expected patient/stay IDs
print("\nUNIQUE ED STAYS")
print("ED stay records:", edstays["stay_id"].nunique())
print("Triage stay records:", triage["stay_id"].nunique())
print("Vital-sign stay records:", vitalsign["stay_id"].nunique())

# 4. Examine the recorded visit times
edstays["intime"] = pd.to_datetime(
    edstays["intime"], errors="coerce"
)
edstays["outtime"] = pd.to_datetime(
    edstays["outtime"], errors="coerce"
)

edstays["ed_stay_hours"] = (
    edstays["outtime"] - edstays["intime"]
).dt.total_seconds() / 3600

print("\nTOTAL ED STAY DURATION — DATA CHECK")
print(edstays["ed_stay_hours"].describe().round(2))

print(
    "\nMissing arrival times:",
    edstays["intime"].isna().sum()
)
print(
    "Missing departure times:",
    edstays["outtime"].isna().sum()
)
print(
    "Zero or negative durations:",
    (edstays["ed_stay_hours"] <= 0).sum()
)


In [ ]:
# ER Band — Step 5: Prepare the training table

import pandas as pd
import numpy as np

# One row per emergency department stay
data = edstays.merge(
    triage,
    on=["subject_id", "stay_id"],
    how="left",
    validate="one_to_one"
)

# Target: total time from ED arrival to ED departure, in hours
data["intime"] = pd.to_datetime(data["intime"], errors="coerce")
data["outtime"] = pd.to_datetime(data["outtime"], errors="coerce")

data["ed_stay_hours"] = (
    data["outtime"] - data["intime"]
).dt.total_seconds() / 3600

# Features available at or near triage
data["arrival_hour"] = data["intime"].dt.hour
data["arrival_day_of_week"] = data["intime"].dt.dayofweek

numeric_features = [
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp",
    "acuity",
    "arrival_hour",
    "arrival_day_of_week"
]

categorical_features = ["arrival_transport"]

# Ensure numeric columns contain numbers, with invalid values
# converted to missing values for later imputation.
for col in numeric_features:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Keep valid target values and required patient identifiers.
data = data.dropna(
    subset=["subject_id", "intime", "outtime", "ed_stay_hours"]
).copy()

data = data[data["ed_stay_hours"] > 0].copy()

feature_columns = numeric_features + categorical_features

X = data[feature_columns].copy()
y = data["ed_stay_hours"].copy()

print("Training table created.")
print("Number of visits:", len(data))
print("Number of different patients:", data["subject_id"].nunique())
print("Number of input features:", len(feature_columns))

print("\nTarget summary — total ED stay in hours:")
print(y.describe().round(2))

print("\nMissing input values:")
print(X.isna().sum())

print("\nExample features:")
print(X.head())

print("\nTarget examples:")
print(y.head())


In [ ]:
# ER Band — Step 6: Train a baseline waiting-time model

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Split by patient, rather than by visit.
# This helps prevent visits from the same patient appearing
# in both training and test data.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=data["subject_id"])
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Prepare numerical and categorical features.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Random Forest regression model.
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
model.fit(X_train, y_train)

# Predict on held-out patients
predictions = model.predict(X_test)

# Compare with a simple baseline that always predicts
# the median duration from the training set.
baseline_predictions = np.full(
    len(y_test), y_train.median()
)

model_mae = mean_absolute_error(y_test, predictions)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
model_rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("TRAINING COMPLETE")
print("Training visits:", len(X_train))
print("Test visits:", len(X_test))

print("\nEVALUATION")
print(f"Model MAE:          {model_mae:.2f} hours")
print(f"Median baseline MAE: {baseline_mae:.2f} hours")
print(f"Model RMSE:         {model_rmse:.2f} hours")


In [ ]:
# ER Band — Step 7: Investigate prediction errors

import pandas as pd
import numpy as np

results = pd.DataFrame({
    "actual_hours": np.asarray(y_test),
    "predicted_hours": predictions,
    "baseline_hours": baseline_predictions
})

results["model_error_hours"] = (
    results["predicted_hours"] - results["actual_hours"]
)

results["model_absolute_error"] = (
    results["model_error_hours"].abs()
)

results["baseline_absolute_error"] = (
    results["baseline_hours"] - results["actual_hours"]
).abs()

model_mae = results["model_absolute_error"].mean()
baseline_mae = results["baseline_absolute_error"].mean()

print("MODEL DIAGNOSTICS")
print(f"Model MAE:    {model_mae:.4f} hours")
print(f"Baseline MAE: {baseline_mae:.4f} hours")
print(
    f"MAE improvement: {baseline_mae - model_mae:.4f} hours"
)

print("\nActual test-set durations:")
print(
    results["actual_hours"]
    .describe(percentiles=[0.5, 0.9, 0.95])
    .round(2)
)

print("\n10 predictions with the largest errors:")
display(
    results.sort_values(
        "model_absolute_error", ascending=False
    ).head(10).round(2)
)


In [ ]:
# ER Band — Step 8: Investigate unusually long stays

# Display visits lasting 24 hours or more.
# We investigate these records rather than automatically
# removing them.

columns_to_inspect = [
    "ed_stay_hours",
    "disposition",
    "arrival_transport",
    "acuity",
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp",
    "chiefcomplaint"
]

long_stays = (
    data.loc[
        data["ed_stay_hours"] >= 24,
        columns_to_inspect
    ]
    .sort_values("ed_stay_hours", ascending=False)
)

print("Visits lasting 24 hours or longer:")
print("Number of visits:", len(long_stays))

display(long_stays.round(2))


In [ ]:
# ER Band — Step 9: Compare errors by actual stay duration

results = results.copy()

# Group visits by their actual duration
results["stay_group"] = pd.cut(
    results["actual_hours"],
    bins=[0, 8, 24, float("inf")],
    labels=["Up to 8 hours", "Over 8 to 24 hours", "Over 24 hours"]
)

# Compare the model against the median baseline
error_summary = results.groupby(
    "stay_group", observed=False
).apply(
    lambda group: pd.Series({
        "test_visits": len(group),
        "model_MAE_hours": group["model_absolute_error"].mean(),
        "baseline_MAE_hours": group["baseline_absolute_error"].mean()
    }),
    include_groups=False
)

print("ERRORS BY ACTUAL STAY DURATION")
display(error_summary.round(2))


In [ ]:

# ER Band — Step 10: Inspect triage acuity labels

print("Recorded acuity levels:")
print(
    data["acuity"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPercentage of records at each level:")
print(
    (data["acuity"].value_counts(normalize=True) * 100)
    .sort_index()
    .round(1)
)

print("\nMissing acuity labels:", data["acuity"].isna().sum())

# Proposed demonstration target:
# 1 = high initial acuity (levels 1 or 2)
# 0 = other initial acuity levels (3, 4 or 5)

labelled = data.dropna(subset=["acuity"]).copy()
labelled = labelled[labelled["acuity"].isin([1, 2, 3, 4, 5])].copy()

labelled["high_initial_acuity"] = (
    labelled["acuity"] <= 2
).astype(int)

print("\nProposed target distribution:")
print(
    labelled["high_initial_acuity"]
    .value_counts()
    .rename(index={0: "Levels 3–5", 1: "Levels 1–2"})
)

print("\nTarget percentages:")
print(
    (labelled["high_initial_acuity"].value_counts(normalize=True) * 100)
    .rename(index={0: "Levels 3–5", 1: "Levels 1–2"})
    .round(1)
)


In [ ]:

# ER Band — Step 11: Train the initial acuity classifier

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score
)

# 1. Prepare records with valid triage labels.
labelled = data.dropna(subset=["acuity"]).copy()

labelled = labelled[
    labelled["acuity"].isin([1, 2, 3, 4, 5])
].copy()

# Target: 1 = high initial acuity; 0 = other acuity.
labelled["high_initial_acuity"] = (
    labelled["acuity"] <= 2
).astype(int)

# 2. Select features recorded at or near triage.
# Deliberately exclude acuity and all post-visit outcomes.

numeric_features_cls = [
    "temperature",
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp",
    "pain"
]

categorical_features_cls = [
    "arrival_transport",
    "chiefcomplaint"
]

feature_columns_cls = (
    numeric_features_cls + categorical_features_cls
)

X_cls = labelled[feature_columns_cls].copy()
y_cls = labelled["high_initial_acuity"].copy()

# Convert numeric features safely.
for col in numeric_features_cls:
    X_cls[col] = pd.to_numeric(X_cls[col], errors="coerce")

print("Training examples:", len(labelled))
print("High initial acuity:", int(y_cls.sum()))
print("Other acuity:", int((y_cls == 0).sum()))

# 3. Split by patient to reduce leakage between visits.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X_cls,
        y_cls,
        groups=labelled["subject_id"]
    )
)

X_train_cls = X_cls.iloc[train_idx]
X_test_cls = X_cls.iloc[test_idx]
y_train_cls = y_cls.iloc[train_idx]
y_test_cls = y_cls.iloc[test_idx]

print("\nTraining cases:", len(X_train_cls))
print("Test cases:", len(X_test_cls))
print("Training class counts:")
print(y_train_cls.value_counts().sort_index())
print("Test class counts:")
print(y_test_cls.value_counts().sort_index())

# 4. Preprocess missing values and categorical fields.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features_cls),
    ("categorical", categorical_pipeline, categorical_features_cls)
])

# 5. Train the Random Forest classifier.
classifier = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ))
])

classifier.fit(X_train_cls, y_train_cls)

# 6. Evaluate on held-out patients.
pred_cls = classifier.predict(X_test_cls)
prob_cls = classifier.predict_proba(X_test_cls)[:, 1]

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test_cls,
    pred_cls,
    labels=[0, 1],
    target_names=[
        "Other acuity (levels 3–4)",
        "High initial acuity (levels 1–2)"
    ],
    zero_division=0
))

print("Confusion matrix (actual rows, predicted columns):")
print(confusion_matrix(y_test_cls, pred_cls, labels=[0, 1]))

print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test_cls, pred_cls), 3)
)

# Probability metrics require both classes in the test set.
if y_test_cls.nunique() == 2:
    print("ROC-AUC:", round(roc_auc_score(y_test_cls, prob_cls), 3))
    print(
        "Average precision:",
        round(average_precision_score(y_test_cls, prob_cls), 3)
    )
else:
    print("ROC-AUC and average precision need both classes in the test set.")

# 7. Compare against a simple majority-class baseline.
baseline_cls = DummyClassifier(strategy="most_frequent")
baseline_cls.fit(X_train_cls, y_train_cls)
baseline_pred_cls = baseline_cls.predict(X_test_cls)

print(
    "\nMajority baseline balanced accuracy:",
    round(balanced_accuracy_score(y_test_cls, baseline_pred_cls), 3)
)


In [ ]:
# ER Band — Step 12: Evaluate performance across five folds

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

# Preserve patient separation while keeping class proportions
# as balanced as possible across folds.
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
    "recall": "recall",
    "precision": "precision"
}

cv_results = cross_validate(
    classifier,
    X_cls,
    y_cls,
    groups=labelled["subject_id"],
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    error_score="raise"
)

# Summarise each fold
fold_results = pd.DataFrame({
    "ROC-AUC": cv_results["test_roc_auc"],
    "Average precision": cv_results["test_average_precision"],
    "Balanced accuracy": cv_results["test_balanced_accuracy"],
    "Recall": cv_results["test_recall"],
    "Precision": cv_results["test_precision"]
}, index=[f"Fold {i}" for i in range(1, 6)])

print("RESULTS BY FOLD")
display(fold_results.round(3))

print("\nMEAN PERFORMANCE")
display(fold_results.mean().rename("Mean").to_frame().round(3))

print("\nVARIATION BETWEEN FOLDS (standard deviation)")
display(fold_results.std().rename("Std dev").to_frame().round(3))


In [ ]:

# ER Band — Step 13: Inspect wearable-relevant time-series data

import pandas as pd
import numpy as np

# Join observations to each ED visit's arrival and departure times.
visit_times = edstays[["stay_id", "intime", "outtime"]].copy()

for col in ["intime", "outtime"]:
    visit_times[col] = pd.to_datetime(
        visit_times[col], errors="coerce"
    )

observations = vitalsign.merge(
    visit_times,
    on="stay_id",
    how="left",
    validate="many_to_one"
)

observations["charttime"] = pd.to_datetime(
    observations["charttime"], errors="coerce"
)

observations = observations.dropna(
    subset=["stay_id", "intime", "charttime"]
).copy()

# Work out when each observation was recorded.
observations["minutes_after_arrival"] = (
    observations["charttime"] - observations["intime"]
).dt.total_seconds() / 60

observations["minutes_before_departure"] = (
    observations["outtime"] - observations["charttime"]
).dt.total_seconds() / 60

print("TIME-SERIES DATA SUMMARY")
print("Observation records:", len(observations))
print("Visits represented:", observations["stay_id"].nunique())

# Count observations per visit.
counts = observations.groupby("stay_id").size()

print("\nOBSERVATIONS PER VISIT")
print(counts.describe().round(2))
print("Visits with 1 observation:", int((counts == 1).sum()))
print("Visits with 2+ observations:", int((counts >= 2).sum()))
print("Visits with 3+ observations:", int((counts >= 3).sum()))

# Check whether observation times fall inside the recorded ED stay.
outside = (
    (observations["minutes_after_arrival"] < 0) |
    (observations["minutes_before_departure"] < 0)
)

print("\nObservations outside recorded stay:", int(outside.sum()))

# Check the completeness of measurements that might be relevant
# to a future monitoring prototype.
vital_columns = [
    "temperature", "heartrate", "resprate",
    "o2sat", "sbp", "dbp", "pain"
]

print("\nMISSING OBSERVATIONS (%)")
print(
    (observations[vital_columns].isna().mean() * 100)
    .round(1)
    .sort_values(ascending=False)
)

# Inspect one visit with a relatively long observation history.
if not counts.empty:
    example_stay = counts.idxmax()

    example = observations[
        observations["stay_id"] == example_stay
    ].sort_values("charttime")

    print("\nEXAMPLE VISIT WITH MOST OBSERVATIONS")
    print("Stay ID:", example_stay)

    display(
        example[
            ["charttime", "minutes_after_arrival"] + vital_columns
        ].round(2)
    )


In [ ]:

# ER Band — Step 14: Check timestamps and plot an observation timeline

import pandas as pd
import matplotlib.pyplot as plt

# 1. Identify observations recorded outside the ED stay.
before_arrival = observations["charttime"] < observations["intime"]
after_departure = observations["charttime"] > observations["outtime"]

print("OBSERVATIONS OUTSIDE THE RECORDED STAY")
print("Before arrival:", int(before_arrival.sum()))
print("After departure:", int(after_departure.sum()))

# Inspect a few examples instead of deleting them automatically.
outside_stay = observations.loc[
    before_arrival | after_departure,
    [
        "stay_id", "charttime", "intime", "outtime",
        "minutes_after_arrival", "minutes_before_departure"
    ]
].copy()

display(outside_stay.head(10))

# 2. For this exploratory plot, retain only observations
# within the recorded ED stay.
inside_stay = observations.loc[
    (observations["charttime"] >= observations["intime"]) &
    (observations["charttime"] <= observations["outtime"])
].copy()

inside_stay["minutes_after_arrival"] = (
    inside_stay["charttime"] - inside_stay["intime"]
).dt.total_seconds() / 60

inside_stay = inside_stay.sort_values(
    ["stay_id", "charttime"]
)

print("\nFILTERED TIMELINE DATA")
print("Original observations:", len(observations))
print("Observations within stay:", len(inside_stay))
print("Visits retained:", inside_stay["stay_id"].nunique())

# 3. Select the visit with the most in-stay observations.
observation_counts = inside_stay.groupby("stay_id").size()

if observation_counts.empty:
    print("No observations remain within the recorded stays.")
else:
    example_stay = observation_counts.idxmax()

    example = inside_stay[
        inside_stay["stay_id"] == example_stay
    ].copy()

    print("\nExample stay ID:", example_stay)
    print("Observations in this stay:", len(example))

    # 4. Plot changes over time.
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    plots = [
        ("heartrate", "Heart rate (beats/min)"),
        ("o2sat", "Oxygen saturation (%)"),
        ("resprate", "Respiratory rate (breaths/min)")
    ]

    for ax, (column, title) in zip(axes.flat[:3], plots):
        valid = example.dropna(
            subset=["minutes_after_arrival", column]
        )

        ax.plot(
            valid["minutes_after_arrival"],
            valid[column],
            marker="o"
        )
        ax.set_title(title)
        ax.set_xlabel("Minutes after ED arrival")
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.3)

    # Blood pressure: plot systolic and diastolic separately.
    ax = axes.flat[3]

    for column, label in [
        ("sbp", "Systolic"),
        ("dbp", "Diastolic")
    ]:
        valid = example.dropna(
            subset=["minutes_after_arrival", column]
        )
        ax.plot(
            valid["minutes_after_arrival"],
            valid[column],
            marker="o",
            label=label
        )

    ax.set_title("Blood pressure (mmHg)")
    ax.set_xlabel("Minutes after ED arrival")
    ax.set_ylabel("Pressure (mmHg)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.suptitle(
        "ER Band — Example ED Observation Timeline",
        fontsize=14
    )
    plt.tight_layout()
    plt.show()


In [ ]:

# ER Band — Step 15: Create patient observation trend features

import numpy as np
import pandas as pd

# Keep only observations recorded within the ED stay.
timeline = observations.loc[
    observations["charttime"].between(
        observations["intime"],
        observations["outtime"],
        inclusive="both"
    )
].copy()

timeline = timeline.sort_values(
    ["stay_id", "charttime"]
)

# Measurements to examine in this first experiment.
# Temperature is excluded until its units are checked.
vital_columns = [
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp"
]

# Convert measurements to numeric values.
for col in vital_columns:
    timeline[col] = pd.to_numeric(
        timeline[col], errors="coerce"
    )

# Create one summary row per ED visit.
summary = timeline.groupby("stay_id").agg(
    observation_count=("charttime", "size"),
    first_observation=("charttime", "min"),
    last_observation=("charttime", "max")
)

summary["observation_window_minutes"] = (
    summary["last_observation"] -
    summary["first_observation"]
).dt.total_seconds() / 60

# Summarise the first and last recorded values for each vital.
for col in vital_columns:
    values = timeline.groupby("stay_id")[col].agg(
        first="first",
        last="last"
    )

    summary[f"{col}_first"] = values["first"]
    summary[f"{col}_last"] = values["last"]

    # Positive means the last recorded value was higher.
    summary[f"{col}_change"] = (
        values["last"] - values["first"]
    )

print("FEATURE TABLE CREATED")
print("ED visits with in-stay observations:", len(summary))
print("Total in-stay observations:", len(timeline))

print("\nFEATURES AVAILABLE")
print(summary.columns.tolist())

print("\nEXAMPLE VISITS")
display(summary.head(10).round(2))

print("\nMISSING FIRST/LAST VALUES")
display(
    summary[
        [f"{col}_first" for col in vital_columns] +
        [f"{col}_last" for col in vital_columns]
    ].isna().sum().to_frame("missing_count")
)


In [ ]:

# ER Band — Step 16: Create first-hour prediction features

import pandas as pd
import numpy as np

vital_columns = [
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp"
]

# Keep only readings from the first 60 minutes of the ED stay.
first_hour = timeline.loc[
    timeline["minutes_after_arrival"].between(
        0, 60, inclusive="both"
    )
].copy()

first_hour = first_hour.sort_values(
    ["stay_id", "charttime"]
)

for col in vital_columns:
    first_hour[col] = pd.to_numeric(
        first_hour[col], errors="coerce"
    )

# Count readings recorded in that first hour.
count_features = (
    first_hour.groupby("stay_id")
    .size()
    .rename("first_hour_observation_count")
)

feature_parts = [count_features]

# Calculate features using ONLY readings from the first hour.
for col in vital_columns:
    stats = first_hour.groupby("stay_id")[col].agg(
        ["first", "last", "min", "max", "mean"]
    )

    stats.columns = [
        f"{col}_{stat}_first_hour"
        for stat in stats.columns
    ]

    feature_parts.append(stats)

first_hour_features = pd.concat(
    feature_parts, axis=1
).reset_index()

# Attach the recorded ED stay duration as our eventual target.
# Rename triage measurements to distinguish them from later readings.
base = data[
    [
        "stay_id", "subject_id", "ed_stay_hours",
        "arrival_transport", "acuity",
        "heartrate", "resprate", "o2sat", "sbp", "dbp"
    ]
].copy()

base = base.rename(columns={
    "heartrate": "triage_heartrate",
    "resprate": "triage_resprate",
    "o2sat": "triage_o2sat",
    "sbp": "triage_sbp",
    "dbp": "triage_dbp",
    "acuity": "triage_acuity"
})

base["triage_acuity"] = base["triage_acuity"].apply(
    lambda x: str(int(x)) if pd.notna(x) else np.nan
)

first_hour_dataset = first_hour_features.merge(
    base,
    on="stay_id",
    how="inner",
    validate="one_to_one"
)

# A prediction at minute 60 only makes sense for visits
# that are still in the ED after that point.
eligible = first_hour_dataset.loc[
    first_hour_dataset["ed_stay_hours"] > 1
].copy()

eligible["remaining_ed_hours_at_60m"] = (
    eligible["ed_stay_hours"] - 1
)

print("FIRST-HOUR DATASET")
print("Visits with at least one first-hour reading:",
      len(first_hour_dataset))
print("Visits still in the ED after 60 minutes:",
      len(eligible))
print("Unique patients in eligible visits:",
      eligible["subject_id"].nunique())

if not eligible.empty:
    print("\nRemaining ED duration after minute 60 (hours):")
    print(
        eligible["remaining_ed_hours_at_60m"]
        .describe()
        .round(2)
    )

    print("\nFirst-hour observation count:")
    print(
        eligible["first_hour_observation_count"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print("\nExample first-hour features:")
    display(first_hour_dataset.head(5).round(2))


In [ ]:

# ER Band — Step 17: Predict remaining ED stay after 60 minutes

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Define only inputs available from arrival and the first hour.
numeric_features = ["first_hour_observation_count"]

for vital in ["heartrate", "resprate", "o2sat", "sbp", "dbp"]:
    for statistic in ["first", "last", "min", "max", "mean"]:
        numeric_features.append(
            f"{vital}_{statistic}_first_hour"
        )

numeric_features += [
    "triage_heartrate",
    "triage_resprate",
    "triage_o2sat",
    "triage_sbp",
    "triage_dbp"
]

categorical_features = [
    "arrival_transport",
    "triage_acuity"
]

feature_columns = numeric_features + categorical_features

# 2. Prepare inputs and target.
# Target: hours remaining AFTER the first 60 minutes.
X = eligible[feature_columns].copy()
y = eligible["remaining_ed_hours_at_60m"].copy()

for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")

print("Visits available:", len(X))
print("Unique patients:", eligible["subject_id"].nunique())

# 3. Split by patient so the same patient doesn't appear
# in both the training and test sets.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=eligible["subject_id"])
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("\nTraining visits:", len(X_train))
print("Test visits:", len(X_test))
print(
    "Training patients:",
    eligible.iloc[train_idx]["subject_id"].nunique()
)
print(
    "Test patients:",
    eligible.iloc[test_idx]["subject_id"].nunique()
)

# 4. Preprocess data.
preprocessor = ColumnTransformer([
    ("numeric",
     SimpleImputer(strategy="median"),
     numeric_features),

    ("categorical",
     Pipeline([
         ("imputer", SimpleImputer(strategy="most_frequent")),
         ("encoder", OneHotEncoder(handle_unknown="ignore"))
     ]),
     categorical_features)
])

# 5. Train a modest Random Forest model.
model_first_hour = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

model_first_hour.fit(X_train, y_train)

# 6. Evaluate on patients held out from training.
predictions_first_hour = model_first_hour.predict(X_test)

# Never predict a negative remaining duration.
predictions_first_hour = np.maximum(
    predictions_first_hour, 0
)

# Baseline: always predict the training-set median.
baseline_predictions = np.full(
    len(y_test), y_train.median()
)

model_mae = mean_absolute_error(
    y_test, predictions_first_hour
)

baseline_mae = mean_absolute_error(
    y_test, baseline_predictions
)

model_rmse = np.sqrt(
    mean_squared_error(y_test, predictions_first_hour)
)

print("\nFIRST-HOUR MODEL RESULTS")
print(f"Model MAE:           {model_mae:.2f} hours")
print(f"Median baseline MAE: {baseline_mae:.2f} hours")
print(f"Model RMSE:          {model_rmse:.2f} hours")
print(
    f"MAE improvement:     {baseline_mae - model_mae:.2f} hours"
)


In [ ]:

# ER Band — Step 18: Inspect first-hour model errors

first_hour_results = eligible.iloc[test_idx][
    ["stay_id", "subject_id", "ed_stay_hours"]
].copy()

first_hour_results["actual_remaining_hours"] = (
    y_test.to_numpy()
)

first_hour_results["predicted_remaining_hours"] = (
    predictions_first_hour
)

first_hour_results["baseline_remaining_hours"] = (
    baseline_predictions
)

first_hour_results["model_abs_error"] = (
    first_hour_results["actual_remaining_hours"]
    - first_hour_results["predicted_remaining_hours"]
).abs()

first_hour_results["baseline_abs_error"] = (
    first_hour_results["actual_remaining_hours"]
    - first_hour_results["baseline_remaining_hours"]
).abs()

print("TEN LARGEST MODEL ERRORS")
display(
    first_hour_results.sort_values(
        "model_abs_error", ascending=False
    ).head(10).round(2)
)

print("\nOVERALL COMPARISON")
print(
    "Model MAE:",
    round(first_hour_results["model_abs_error"].mean(), 2)
)

print(
    "Baseline MAE:",
    round(first_hour_results["baseline_abs_error"].mean(), 2)
)


In [ ]:

# ER Band — Step 19: Test a log-transformed target

import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.base import clone

# Train the same model, but transform the target during fitting.
log_model = TransformedTargetRegressor(
    regressor=clone(model_first_hour),
    func=np.log1p,
    inverse_func=np.expm1
)

log_model.fit(X_train, y_train)

# Convert predictions back to hours.
log_predictions = np.maximum(
    log_model.predict(X_test), 0
)

# Compare all three approaches on the same test visits.
comparison = pd.DataFrame({
    "Actual hours remaining": np.asarray(y_test),
    "Random Forest": predictions_first_hour,
    "Log-target Random Forest": log_predictions,
    "Median baseline": baseline_predictions
})

rows = []

for name in [
    "Random Forest",
    "Log-target Random Forest",
    "Median baseline"
]:
    errors = (
        comparison["Actual hours remaining"] -
        comparison[name]
    )

    rows.append({
        "Model": name,
        "MAE_hours": mean_absolute_error(
            comparison["Actual hours remaining"],
            comparison[name]
        ),
        "RMSE_hours": np.sqrt(
            mean_squared_error(
                comparison["Actual hours remaining"],
                comparison[name]
            )
        )
    })

print("MODEL COMPARISON ON THE SAME TEST SET")
display(pd.DataFrame(rows).round(2))

print("\nLOG-TARGET PREDICTIONS FOR THE LARGEST ACTUAL STAYS")
display(
    comparison.sort_values(
        "Actual hours remaining", ascending=False
    ).head(10).round(2)
)


In [ ]:

# ER Band — Step 20: Exploratory anomaly detection
# This detects statistically unusual observations, not clinical urgency.

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# Use observations within the recorded ED stay only.
trend_data = timeline.copy().sort_values(
    ["subject_id", "stay_id", "charttime"]
)

vitals = ["heartrate", "resprate", "o2sat", "sbp", "dbp"]

for col in vitals:
    trend_data[col] = pd.to_numeric(
        trend_data[col], errors="coerce"
    )

# Features available at each recorded observation:
# current measurements + changes from the previous observation.
feature_cols = list(vitals)

for col in vitals:
    change_col = f"{col}_change_from_previous"
    trend_data[change_col] = (
        trend_data.groupby("stay_id")[col].diff()
    )
    feature_cols.append(change_col)

# Keep rows containing at least one current vital measurement.
trend_data = trend_data.dropna(
    subset=vitals, how="all"
).copy()

X = trend_data[feature_cols]
groups = trend_data["subject_id"]

# Hold out entire patients rather than randomly splitting rows.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

train_data = trend_data.iloc[train_idx].copy()
test_data = trend_data.iloc[test_idx].copy()

# Fit preprocessing using training observations only.
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    imputer.fit_transform(X_train)
)

X_test_scaled = scaler.transform(
    imputer.transform(X_test)
)

# Train an unsupervised anomaly detector.
detector = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42
)

detector.fit(X_train_scaled)

# Lower scores indicate observations more unusual
# relative to the fitted training distribution.
train_scores = detector.decision_function(X_train_scaled)
test_scores = detector.decision_function(X_test_scaled)

# Demo threshold: flag the lowest-scoring 5% of training
# observations. This percentage is arbitrary, not clinical.
threshold = np.percentile(train_scores, 5)

test_data["anomaly_score"] = test_scores
test_data["demo_anomaly_flag"] = test_scores < threshold

print("ANOMALY DETECTOR")
print("Training observations:", len(X_train))
print("Test observations:", len(X_test))
print(
    "Training patients:",
    train_data["subject_id"].nunique()
)
print(
    "Test patients:",
    test_data["subject_id"].nunique()
)
print("\nDemo flags:", int(test_data["demo_anomaly_flag"].sum()))
print(
    "Percentage of test observations flagged:",
    round(100 * test_data["demo_anomaly_flag"].mean(), 1), "%"
)

# Inspect the most unusual test observations.
display(
    test_data.sort_values("anomaly_score")[
        [
            "stay_id", "charttime",
            "heartrate", "resprate", "o2sat",
            "sbp", "dbp", "anomaly_score",
            "demo_anomaly_flag"
        ]
    ].head(10).round(3)
)


In [ ]:

# ER Band — Step 21: Sensor-aligned anomaly detection
# Demonstration only: not a clinical deterioration detector.

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# Use only measurements the proposed optical sensor
# can experiment with.
sensor_features = [
    "heartrate",
    "o2sat",
    "heartrate_change_from_previous",
    "o2sat_change_from_previous"
]

X_sensor = trend_data[sensor_features].copy()
groups = trend_data["subject_id"]

# Keep all visits from the same patient in one split.
splitter_sensor = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_sensor_idx, test_sensor_idx = next(
    splitter_sensor.split(X_sensor, groups=groups)
)

X_train_sensor = X_sensor.iloc[train_sensor_idx]
X_test_sensor = X_sensor.iloc[test_sensor_idx]

# Learn imputation and scaling from training data only.
imputer_sensor = SimpleImputer(strategy="median")
scaler_sensor = StandardScaler()

X_train_scaled = scaler_sensor.fit_transform(
    imputer_sensor.fit_transform(X_train_sensor)
)

X_test_scaled = scaler_sensor.transform(
    imputer_sensor.transform(X_test_sensor)
)

# Train the detector.
detector_sensor = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42
)

detector_sensor.fit(X_train_scaled)

train_scores = detector_sensor.decision_function(
    X_train_scaled
)
test_scores = detector_sensor.decision_function(
    X_test_scaled
)

# Use a demonstration threshold based on training scores.
threshold_sensor = np.percentile(train_scores, 5)

flags_sensor = test_scores < threshold_sensor

# Prepare results suitable for inspection or a demo dashboard.
sensor_results = trend_data.iloc[test_sensor_idx][
    [
        "subject_id",
        "stay_id",
        "charttime",
        "heartrate",
        "o2sat"
    ]
].copy()

sensor_results["anomaly_score"] = test_scores
sensor_results["demo_flag"] = flags_sensor

print("SENSOR-ALIGNED DETECTOR")
print("Training observations:", len(X_train_sensor))
print("Test observations:", len(X_test_sensor))
print(
    "Training patients:",
    trend_data.iloc[train_sensor_idx]["subject_id"].nunique()
)
print(
    "Test patients:",
    trend_data.iloc[test_sensor_idx]["subject_id"].nunique()
)
print("Observations flagged:", int(flags_sensor.sum()))
print(
    "Flagged percentage:",
    round(100 * flags_sensor.mean(), 1), "%"
)

print("\nMost unusual test observations:")
display(
    sensor_results.sort_values(
        "anomaly_score"
    ).head(10).round(3)
)


In [ ]:

# ER Band — Step 22: Create a review-ready output table
# This is a demonstration interface, not a clinical alert system.

import numpy as np
import pandas as pd

review = sensor_results.copy()

# Make missing values explicit.
review["missing_heartrate"] = review["heartrate"].isna()
review["missing_o2sat"] = review["o2sat"].isna()

review["missing_sensor_data"] = (
    review["missing_heartrate"] |
    review["missing_o2sat"]
)

review["demo_flag"] = (
    review["demo_flag"].fillna(False).astype(bool)
)

# Use language that describes the algorithm's output
# without making a claim about patient urgency.
review["dashboard_message"] = np.select(
    [
        review["missing_sensor_data"] & review["demo_flag"],
        review["missing_sensor_data"],
        review["demo_flag"]
    ],
    [
        "Unusual pattern; check missing readings",
        "Incomplete sensor data; check measurement",
        "Unusual pattern; review observation"
    ],
    default="No anomaly flag in this demonstration"
)

flagged = review[review["demo_flag"]].copy()

print("DATA QUALITY AND FLAG SUMMARY")
print("Test observations:", len(review))
print("Flagged observations:", len(flagged))
print(
    "Unique stays with flags:",
    flagged["stay_id"].nunique()
)
print(
    "Unique patients with flags:",
    flagged["subject_id"].nunique()
)
print(
    "Flagged observations with missing HR or SpO2:",
    int(flagged["missing_sensor_data"].sum())
)

print("\nREVIEW TABLE")
display(
    review.sort_values("anomaly_score")[
        [
            "stay_id",
            "charttime",
            "heartrate",
            "o2sat",
            "anomaly_score",
            "missing_sensor_data",
            "dashboard_message"
        ]
    ].head(15).round(3)
)


In [ ]:

# ER Band — Step 23: Visualise a flagged observation timeline
# Demonstration using held-out, de-identified ED observations.

import matplotlib.pyplot as plt
import pandas as pd

# Rebuild the test-set timeline and attach model scores.
test_timeline = trend_data.iloc[test_sensor_idx].copy()

test_timeline["anomaly_score"] = test_scores
test_timeline["demo_flag"] = flags_sensor

# Find the stay with the largest number of flagged observations.
flag_counts = (
    test_timeline[test_timeline["demo_flag"]]
    .groupby("stay_id")
    .size()
    .sort_values(ascending=False)
)

if flag_counts.empty:
    print("No flagged observations available to plot.")
else:
    example_stay = flag_counts.index[0]

    plot_data = (
        test_timeline[
            test_timeline["stay_id"] == example_stay
        ]
        .sort_values("charttime")
        .copy()
    )

    print("Example stay ID:", example_stay)
    print("Observations in this stay:", len(plot_data))
    print("Observations flagged:", int(plot_data["demo_flag"].sum()))

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Plot heart rate.
    valid_hr = plot_data.dropna(
        subset=["charttime", "heartrate"]
    )
    axes[0].plot(
        valid_hr["charttime"],
        valid_hr["heartrate"],
        marker="o",
        label="Heart rate"
    )

    flagged_hr = valid_hr[valid_hr["demo_flag"]]
    axes[0].scatter(
        flagged_hr["charttime"],
        flagged_hr["heartrate"],
        marker="x",
        s=100,
        label="Flagged observation"
    )

    axes[0].set_title("Heart rate over time")
    axes[0].set_ylabel("Beats per minute")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot oxygen saturation.
    valid_o2 = plot_data.dropna(
        subset=["charttime", "o2sat"]
    )
    axes[1].plot(
        valid_o2["charttime"],
        valid_o2["o2sat"],
        marker="o",
        label="Oxygen saturation"
    )

    flagged_o2 = valid_o2[valid_o2["demo_flag"]]
    axes[1].scatter(
        flagged_o2["charttime"],
        flagged_o2["o2sat"],
        marker="x",
        s=100,
        label="Flagged observation"
    )

    axes[1].set_title("Oxygen saturation over time")
    axes[1].set_ylabel("SpO2 (%)")
    axes[1].set_xlabel("Recorded observation time")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(
        "ER Band — Observation Anomaly Demonstration\n"
        "Flags indicate unusual patterns, not clinical urgency"
    )
    plt.tight_layout()
    plt.show()

    print("\nTimeline data:")
    display(
        plot_data[
            [
                "charttime", "heartrate", "o2sat",
                "anomaly_score", "demo_flag"
            ]
        ].round(3)
    )
